# DATA266 Task 3 - Praveen SID 8511 CycleGAN

This notebook runs the independent Praveen 8511 CycleGAN configuration. Keep this notebook in the same folder as `train.py`, `evaluate.py`, `models.py`, `data.py`, `utils.py`, and `config.json`.

## Model identity

- ResNet generators: 48/96/192 channels and 6 residual blocks
- Nearest-neighbor upsampling followed by convolution
- Spectral-normalized PatchGAN discriminators
- BCE-with-logits adversarial loss
- AdamW, learning rate 1e-4, batch size 2
- Cycle weight 12, identity weight 2
- Seed 8511

In [ ]:
# Run this only if the environment is missing dependencies.
%pip install -q -r requirements.txt

In [ ]:
import json
from pathlib import Path
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
cfg = json.loads(Path('config.json').read_text())
cfg

In [ ]:
# Check model shapes and parameter counts before training.
from models import Generator, Discriminator, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G_A2B = Generator(base=cfg['base_channels'], n_blocks=cfg['generator_res_blocks']).to(device)
G_B2A = Generator(base=cfg['base_channels'], n_blocks=cfg['generator_res_blocks']).to(device)
D_A = Discriminator(base=cfg['base_channels'], use_spectral_norm=cfg['use_spectral_norm']).to(device)
D_B = Discriminator(base=cfg['base_channels'], use_spectral_norm=cfg['use_spectral_norm']).to(device)
x = torch.randn(1, 3, 256, 256, device=device)
print('Generator output:', G_A2B(x).shape)
print('Discriminator output:', D_A(x).shape)
print('G_A2B parameters:', count_parameters(G_A2B))
print('G_B2A parameters:', count_parameters(G_B2A))
print('D_A parameters:', count_parameters(D_A))
print('D_B parameters:', count_parameters(D_B))

## CPU smoke test

This uses a synthetic RGB dataset and only two updates. It checks the complete forward pass, backward pass, checkpoint creation, logging, and sample generation.

In [ ]:
!python train.py --config config.json --smoke-test --synthetic-smoke-data --output-root ./smoke_outputs --device cpu

## Full training

Set `DATA_ROOT` to the folder containing `monet_jpg/` and `photo_jpg/` or `trainA/` and `trainB/`. The default configuration performs 30,000 updates.

In [ ]:
DATA_ROOT = '/path/to/your/data'  # change this before running
OUTPUT_ROOT = './outputs'
assert DATA_ROOT != '/path/to/your/data', 'Update DATA_ROOT first.'
!python train.py --config config.json --data-root "{DATA_ROOT}" --output-root "{OUTPUT_ROOT}" --device cuda

## Resume training

Use this cell if the GPU session stops.

In [ ]:
# RESUME_CHECKPOINT = './outputs/checkpoints/latest.pt'
# !python train.py --config config.json --data-root "{DATA_ROOT}" --output-root "{OUTPUT_ROOT}" --device cuda --resume "{RESUME_CHECKPOINT}"

## Generate translations from a trained checkpoint

In [ ]:
CHECKPOINT = './outputs/checkpoints/latest.pt'
INPUT_DIR = '/path/to/input/images'  # change this
# !python evaluate.py --checkpoint "{CHECKPOINT}" --input-dir "{INPUT_DIR}" --direction A2B --output-dir ./outputs/inference_A2B --device cuda
# !python evaluate.py --checkpoint "{CHECKPOINT}" --input-dir "{INPUT_DIR}" --direction B2A --output-dir ./outputs/inference_B2A --device cuda

In [ ]:
# Inspect the raw epoch log after training.
import pandas as pd
log_path = Path('outputs/logs/train_raw.jsonl')
if log_path.exists():
    log_df = pd.read_json(log_path, lines=True)
    display(log_df.tail())
    log_df.plot(x='epoch', y=['G', 'D', 'cycle', 'identity'], figsize=(12, 5), title='Praveen 8511 CycleGAN losses')
else:
    print('Run training first; raw log not found:', log_path)

## Required final reporting

Before submission, populate `outputs/metrics_report.csv` with both-direction FID, KID, precision/recall, cycle L1, LPIPS, content cosine, human audit scores, inter-rater agreement, and Kaggle results. Preserve `outputs/logs/train_raw.jsonl` unchanged.